# Bronze Layer Ingestion

#### Data Configuration

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Source and target configuration
VOLUME_PATH = "/Volumes/ecommerce/bronze/raw_data"
CATALOG = "ecommerce"
BRONZE_SCHEMA = "bronze"

# Tables to ingest — maps source parquet filename to target Delta table name
TABLE_MAP = {
    "users":                "users_raw",
    "orders":               "orders_raw",
    "order_items":          "order_items_raw",
    "products":             "products_raw",
    "inventory_items":      "inventory_items_raw",
    "distribution_centers": "distribution_centers_raw",
    "events":               "events_raw",
}

# Set the catalog context
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"Catalog: {CATALOG}")
print(f"Schema:  {BRONZE_SCHEMA}")
print(f"Tables to ingest: {len(TABLE_MAP)}")

Catalog: ecommerce
Schema:  bronze
Tables to ingest: 7


#### Data Ingestion

In [0]:
import time

def ingest_to_bronze(source_name: str, target_table: str) -> dict:
    """
    Read Parquet from Volume, add lineage metadata, write as managed Delta table.
    Returns ingestion metrics.
    """
    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{target_table}"
    source_path = f"{VOLUME_PATH}/{source_name}.parquet"
    
    start = time.time()
    
    try:
        df = spark.read.parquet(source_path)
        col_count = len(df.columns) + 2

        df = (
            df
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(f"{source_name}.parquet"))
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_table_name)
        )

        row_count = spark.table(full_table_name).count()
        elapsed = round(time.time() - start, 2)

        print(f"  ✓ {full_table_name:<45} {row_count:>10,} rows | {col_count} cols | {elapsed}s")

        return {"table": full_table_name, "rows": row_count, "columns": col_count,
                "elapsed": elapsed, "status": "SUCCESS"}

    except Exception as e:
        elapsed = round(time.time() - start, 2)
        print(f"  ✗ {full_table_name:<45} FAILED | {elapsed}s | {str(e)[:80]}")
        return {"table": full_table_name, "rows": 0, "columns": 0,
                "elapsed": elapsed, "status": "FAILED"}

In [0]:
print("=" * 80)
print("  BRONZE LAYER INGESTION")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80 + "\n")

results = []
for source_name, target_table in TABLE_MAP.items():
    results.append(ingest_to_bronze(source_name, target_table))

succeeded = [r for r in results if r["status"] == "SUCCESS"]
failed = [r for r in results if r["status"] == "FAILED"]
total_rows = sum(r["rows"] for r in succeeded)
total_time = sum(r["elapsed"] for r in results)

print(f"\n{'=' * 80}")
print(f"  COMPLETE — {len(succeeded)}/{len(results)} succeeded | {total_rows:,} total rows | {total_time:.1f}s total")
if failed:
    print(f"  FAILED: {', '.join(r['table'] for r in failed)}")
print(f"{'=' * 80}")

  BRONZE LAYER INGESTION
  2026-07-17 18:25:28

  ✓ ecommerce.bronze.users_raw                       100,000 rows | 18 cols | 3.26s
  ✓ ecommerce.bronze.orders_raw                      124,850 rows | 11 cols | 2.77s
  ✓ ecommerce.bronze.order_items_raw                 181,424 rows | 13 cols | 3.46s
  ✓ ecommerce.bronze.products_raw                     29,120 rows | 11 cols | 3.03s
  ✓ ecommerce.bronze.inventory_items_raw             490,083 rows | 14 cols | 3.13s
  ✓ ecommerce.bronze.distribution_centers_raw             10 rows | 7 cols | 2.79s
  ✓ ecommerce.bronze.events_raw                    2,427,889 rows | 15 cols | 5.7s

  COMPLETE — 7/7 succeeded | 3,353,376 total rows | 24.1s total


#### Data Validation

In [0]:
print("=" * 80)
print("  BRONZE DATA QUALITY CHECKS")
print("=" * 80 + "\n")

# Define primary key column for each table
pk_map = {
    "users_raw": "id",
    "orders_raw": "order_id",
    "order_items_raw": "id",
    "products_raw": "id",
    "inventory_items_raw": "id",
    "distribution_centers_raw": "id",
    "events_raw": "id",
}

for table_name, pk_col in pk_map.items():
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    df = spark.table(full_name)
    
    total = df.count()
    null_pks = df.filter(F.col(pk_col).isNull()).count()
    distinct_pks = df.select(pk_col).distinct().count()
    duplicate_pks = total - distinct_pks
    
    status = "✓" if (null_pks == 0 and duplicate_pks == 0) else "⚠"
    
    print(f"  {status} {table_name:<30} total: {total:>10,} | null PKs: {null_pks} | duplicate PKs: {duplicate_pks:,}")

print(f"\n  Note: Bronze preserves all data as-is. Issues are addressed in Silver.")

  BRONZE DATA QUALITY CHECKS

  ✓ users_raw                      total:    100,000 | null PKs: 0 | duplicate PKs: 0
  ✓ orders_raw                     total:    124,850 | null PKs: 0 | duplicate PKs: 0
  ✓ order_items_raw                total:    181,424 | null PKs: 0 | duplicate PKs: 0
  ✓ products_raw                   total:     29,120 | null PKs: 0 | duplicate PKs: 0
  ✓ inventory_items_raw            total:    490,083 | null PKs: 0 | duplicate PKs: 0
  ✓ distribution_centers_raw       total:         10 | null PKs: 0 | duplicate PKs: 0
  ✓ events_raw                     total:  2,427,889 | null PKs: 0 | duplicate PKs: 0

  Note: Bronze preserves all data as-is. Issues are addressed in Silver.


In [0]:
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.users_raw").limit(5))

id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at,user_geom,_ingested_at,_source_file
60914,Jessica,Watson,jessicawatson@example.net,43,F,Acre,7165 Hall Squares Apt. 968,69980-000,null,Brasil,-8.065346116,-72.87094866,Facebook,2025-09-12T07:24:00.000Z,POINT(-72.87094866 -8.065346116),2026-07-17T18:25:29.658Z,users.parquet
4688,Sabrina,Jones,sabrinajones@example.net,15,F,Acre,6098 Timothy Stream Apt. 682,69980-000,null,Brasil,-8.065346116,-72.87094866,Search,2023-05-01T10:06:00.000Z,POINT(-72.87094866 -8.065346116),2026-07-17T18:25:29.658Z,users.parquet
91933,Paula,Peterson,paulapeterson@example.net,31,F,Acre,8637 Martin Coves,69980-000,null,Brasil,-8.065346116,-72.87094866,Organic,2019-11-10T05:30:00.000Z,POINT(-72.87094866 -8.065346116),2026-07-17T18:25:29.658Z,users.parquet
12362,James,Adams,jamesadams@example.com,20,M,Acre,343 Ryan Pines Suite 714,69980-000,null,Brasil,-8.065346116,-72.87094866,Organic,2024-05-13T17:22:00.000Z,POINT(-72.87094866 -8.065346116),2026-07-17T18:25:29.658Z,users.parquet
76571,Nicole,Jones,nicolejones@example.org,67,F,Acre,25055 Lee Ridge Suite 348,69980-000,null,Brasil,-8.065346116,-72.87094866,Organic,2021-02-12T15:53:00.000Z,POINT(-72.87094866 -8.065346116),2026-07-17T18:25:29.658Z,users.parquet
